# 02 — Preprocessing

**Course:** ADS504 — Machine Learning and Deep Learning for Data Science  
**Dataset:** [Bank Marketing (UCI)](https://archive.ics.uci.edu/dataset/222/bank+marketing)


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight
import joblib

pd.set_option('display.max_columns', 40)


## Load data

Use the same raw file as EDA.


In [ ]:
# Local path from notebooks/. On Colab, upload the file or point this to your Drive path.
DATA_PATH = Path('../data/raw/bank-full.csv')

df = pd.read_csv(DATA_PATH, sep=';')
print('Shape:', df.shape)
df.head()


## Cleaning decisions from EDA

- No duplicate rows and no NaNs to impute
- Keep `unknown` as its own category
- Treat `pdays == -1` as never contacted
- Drop `duration` (leakage)
- Transform skewed numerics instead of deleting outliers


In [ ]:
print('Duplicate rows:', df.duplicated().sum())
print('Nulls:', int(df.isnull().sum().sum()))

df['y_binary'] = (df['y'] == 'yes').astype(int)
print('Yes rate: {:.1f}%'.format(df['y_binary'].mean() * 100))


## Train / validation / test split

Split first so any fit-based transforms use training data only. Stratify to keep the ~12% yes rate.


In [ ]:
y = (df['y'] == 'yes').astype(int)

trainval_idx, test_idx = train_test_split(
    df.index, test_size=0.20, random_state=42, stratify=y
)
train_idx, val_idx = train_test_split(
    trainval_idx, test_size=0.25, random_state=42, stratify=y.loc[trainval_idx]
)

for name, idx in [('train', train_idx), ('val', val_idx), ('test', test_idx)]:
    print(f'{name:5s}: {len(idx):,} rows | yes-rate {y.loc[idx].mean()*100:.1f}%')


## Feature engineering

Create leakage-free features guided by EDA. Cap campaign using the training-set 99th percentile only.


In [ ]:
fe = df.copy()
fe['y_binary'] = y

# Prior-contact flag and cleaned pdays
fe['was_contacted_before'] = (fe['pdays'] != -1).astype(int)
fe['pdays_clean'] = fe['pdays'].replace(-1, 0)

# Skew transforms
fe['balance_slog'] = np.sign(fe['balance']) * np.log1p(fe['balance'].abs())
fe['previous_log'] = np.log1p(fe['previous'])

# Cap extreme campaign counts using TRAIN only, then log
campaign_cap = fe.loc[train_idx, 'campaign'].quantile(0.99)
fe['campaign_capped'] = fe['campaign'].clip(upper=campaign_cap)
fe['campaign_log'] = np.log1p(fe['campaign_capped'])
print('Campaign cap (train 99th pct):', campaign_cap)

# High-conversion months from EDA
high_months = ['mar', 'sep', 'oct', 'dec', 'apr', 'feb']
fe['high_yield_month'] = fe['month'].isin(high_months).astype(int)

# Age bands (same bins as EDA)
fe['age_band'] = pd.cut(
    fe['age'],
    bins=[0, 30, 45, 60, 100],
    labels=['<=30', '31-45', '46-60', '60+']
)

fe.loc[train_idx, ['pdays', 'pdays_clean', 'was_contacted_before', 'balance', 'balance_slog',
                   'campaign', 'campaign_capped', 'campaign_log', 'month', 'high_yield_month', 'age_band']].head()


## Build feature matrix

Drop leakage and raw columns replaced by engineered versions. Keep unknown levels as-is.


In [ ]:
drop_cols = [
    'duration',  # leakage
    'y', 'y_binary',
    'pdays', 'balance', 'campaign', 'previous', 'campaign_capped',
]

X = fe.drop(columns=drop_cols)

num_features = ['age', 'day', 'pdays_clean', 'balance_slog', 'campaign_log', 'previous_log']
cat_features = [
    'job', 'marital', 'education', 'default', 'housing', 'loan',
    'contact', 'month', 'poutcome', 'age_band'
]
flag_features = ['was_contacted_before', 'high_yield_month']

X_train = X.loc[train_idx]
X_val = X.loc[val_idx]
X_test = X.loc[test_idx]
y_train = y.loc[train_idx]
y_val = y.loc[val_idx]
y_test = y.loc[test_idx]

print('X shape:', X.shape)
print('numeric:', num_features)
print('categorical:', cat_features)
print('flags:', flag_features)


## Encode and scale

Fit the transformer on the training set only, then transform val/test.


In [ ]:
preprocess = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features + flag_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_features),
    ],
    remainder='drop'
)

X_train_p = preprocess.fit_transform(X_train)
X_val_p = preprocess.transform(X_val)
X_test_p = preprocess.transform(X_test)

feature_names = preprocess.get_feature_names_out()
print('Prepared shapes:', X_train_p.shape, X_val_p.shape, X_test_p.shape)
print('Feature count:', len(feature_names))


## Class imbalance note

Do not resample the full dataset. Modeling can use `class_weight='balanced'` or oversampling on the training fold only.


In [ ]:
weights = compute_class_weight('balanced', classes=np.array([0, 1]), y=y_train)
print('Suggested class weights: {0: %.3f, 1: %.3f}' % (weights[0], weights[1]))


## Save processed artifacts

Write splits and the fitted preprocessor for modeling.


In [ ]:
OUT_DIR = Path('../data/processed')
OUT_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(preprocess, OUT_DIR / 'preprocessor.joblib')
joblib.dump(feature_names, OUT_DIR / 'feature_names.joblib')

np.save(OUT_DIR / 'X_train.npy', X_train_p)
np.save(OUT_DIR / 'X_val.npy', X_val_p)
np.save(OUT_DIR / 'X_test.npy', X_test_p)
np.save(OUT_DIR / 'y_train.npy', y_train.to_numpy())
np.save(OUT_DIR / 'y_val.npy', y_val.to_numpy())
np.save(OUT_DIR / 'y_test.npy', y_test.to_numpy())

# Also keep tabular train features before encoding (useful for tree models later)
X_train.to_csv(OUT_DIR / 'X_train_raw.csv', index=False)
X_val.to_csv(OUT_DIR / 'X_val_raw.csv', index=False)
X_test.to_csv(OUT_DIR / 'X_test_raw.csv', index=False)
y_train.to_csv(OUT_DIR / 'y_train.csv', index=False)
y_val.to_csv(OUT_DIR / 'y_val.csv', index=False)
y_test.to_csv(OUT_DIR / 'y_test.csv', index=False)

print('Saved to', OUT_DIR.resolve())
print(sorted(p.name for p in OUT_DIR.glob('*') if p.name != '.gitkeep'))


## Summary

- Dropped duration
- Kept unknown categories and engineered a prior-contact flag from pdays
- Log/signed-log transforms for skewed numerics; capped campaign outliers using the training set
- Added high_yield_month and age_band
- Stratified 60/20/20 split; scaler/encoder fit on train only
- Artifacts written to data/processed/ for modeling
